In [12]:
from threading import Thread
import random
import time

counter = 0

def inc_counter():
    global counter
    time.sleep(random.random())
    counter += 1
    time.sleep(random.random())
    print(f"new counter value {counter}")
    time.sleep(random.random())
    print("-------------------------------")

for x in range(10):
    th = Thread(target=inc_counter)
    th.start()

new counter value 5
new counter value 6
new counter value 7
new counter value 8
-------------------------------
new counter value 8
new counter value 9
new counter value 10
-------------------------------
-------------------------------
new counter value 10
-------------------------------
new counter value 10
-------------------------------
-------------------------------
new counter value 10
-------------------------------
-------------------------------
-------------------------------
-------------------------------


# 🏎️ Threads and the "Race Condition" Problem


## 1. The Core Problem: Why this code is broken

If you run this code with 10 threads, it might *look* like it works fine at first. But if you increase that range from 10 to **100,000**, your counter will almost certainly output the wrong final number (e.g., stopping at 94,322 instead of 100,000).

Why does this happen? Because the operation `counter += 1` is **not atomic** (it is not a single, indivisible step). To Python, `counter += 1` actually takes three separate steps:
1. **Read:** Look at the current value of `counter` (e.g., `0`).
2. **Modify:** Add `1` to that value (`0 + 1 = 1`).
3. **Write:** Save the new value back to `counter`.



### The Conflict:
Because threads execute concurrently and switch back and forth rapidly, two threads can look at the counter at the exact same millisecond:
* **Thread 1** reads the counter (it's `0`).
* Before Thread 1 can save its update, Python switches to **Thread 2**.
* **Thread 2** reads the counter (it's still `0`).
* **Thread 2** adds 1 and writes `1`.
* Python switches back to **Thread 1**. **Thread 1** adds 1 and writes `1`.

Two separate threads ran, but the counter only went up by **one** instead of **two**! They overwrote each other's work. This is a **Race Condition**.

---

## 2. The Solution: Threading Locks (`threading.Lock`)

To resolve this, we use a **Lock** (also known as a Mutex). A lock ensures that only **one thread** can execute a specific block of code at a time. All other threads must wait in line until the lock is released.

* **Analogy:** Think of it like a bathroom key at a coffee shop. If you have the key, you are the only one who can enter. Anyone else who wants to use it must wait outside until you come out and hand over the key.

---

## 3. Correct Implementation (Safe Multithreading)

Run the Python cell below to see how to use `threading.Lock()` to make your counter 100% thread-safe.

In [13]:
import threading
import time
import random

counter = 0
# Create a lock object
counter_lock = threading.Lock()

def inc_counter():
    global counter
    
    # Acquire the lock before touching the shared variable
    with counter_lock:
        # Everything inside this 'with' block is now safe.
        # Only ONE thread can be here at any given microsecond.
        counter += 1
        time.sleep(random.random())
        print(f"new counter value {counter}")
        time.sleep(random.random())
        print("-------------------------------")
    # The lock is automatically released here when exiting the 'with' block

# Create and start 10 threads
threads = []
for x in range(10):
    th = threading.Thread(target=inc_counter)
    time.sleep(random.random())
    threads.append(th)
    th.start()

# Ensure all threads finish before moving on
for th in threads:
    th.join()

print(f"Final safe counter value: {counter}")

new counter value 1
-------------------------------
new counter value 2
-------------------------------
new counter value 3
-------------------------------
new counter value 4
-------------------------------
new counter value 5
-------------------------------
new counter value 6
-------------------------------
new counter value 7
-------------------------------
new counter value 8
-------------------------------
new counter value 9
-------------------------------
new counter value 10
-------------------------------
Final safe counter value: 10
